# Метод наименьших квадратов (МНК) — Аппроксимация данных

**Тема:** Аппроксимация данных с шумом полиномами разных степеней и нелинейными функциями  
**Инструменты:** `numpy.linalg.lstsq`, `numpy.polyfit`, `scipy.optimize.curve_fit`

---

## Что такое МНК?

Метод наименьших квадратов находит коэффициенты функции, минимизируя сумму квадратов отклонений между данными и моделью:

$$\min \sum_{i=1}^{n} (y_i - f(x_i))^2$$

Для полиномиальной аппроксимации это сводится к решению системы линейных уравнений через матрицу Вандермонда.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.optimize as sp_optimize

np.random.seed(42)
delta = 1.0
x_raw = np.linspace(2, -15, 12)

## 1. Полином 1-й степени — прямая

$$y = ax + b$$

Матрица системы: столбцы `[x, 1]`

In [ ]:
x = x_raw + delta * (np.random.rand(12) - 0.5)
y = x_raw + delta * (np.random.rand(12) - 0.5)

# Матрица Вандермонда для 1-й степени: [x, 1]
M = np.vstack((x, np.ones(12))).T
s = np.linalg.lstsq(M, y, rcond=None)[0]

x_line = np.linspace(-15, 2, 100)
plt.figure(figsize=(9, 4))
plt.plot(x, y, 'D', label='Данные с шумом')
plt.plot(x_line, s[0]*x_line + s[1], 'r-', lw=2, label=f'y = {s[0]:.2f}x + {s[1]:.2f}')
plt.title('Аппроксимация — 1-я степень (прямая)')
plt.grid(); plt.legend(); plt.show()
print(f"Коэффициенты: a={s[0]:.4f}, b={s[1]:.4f}")

## 2. Полином 2-й степени — парабола

$$y = ax^2 + bx + c$$

Матрица системы: столбцы `[x², x, 1]`

In [ ]:
x = x_raw + delta * (np.random.rand(12) - 0.5)
y = x_raw**2 + delta * (np.random.rand(12) - 0.5)

M = np.vstack((x**2, x, np.ones(12))).T
s = np.linalg.lstsq(M, y, rcond=None)[0]

x_line = np.linspace(-15, 2, 100)
plt.figure(figsize=(9, 4))
plt.plot(x, y, 'D', label='Данные с шумом')
plt.plot(x_line, s[0]*x_line**2 + s[1]*x_line + s[2], 'r-', lw=2,
         label=f'y = {s[0]:.2f}x² + {s[1]:.2f}x + {s[2]:.2f}')
plt.title('Аппроксимация — 2-я степень (парабола)')
plt.grid(); plt.legend(); plt.show()
print(f"Коэффициенты: a={s[0]:.4f}, b={s[1]:.4f}, c={s[2]:.4f}")

## 3. Полином 3-й степени

$$y = ax^3 + bx^2 + cx + d$$

Матрица системы: столбцы `[x³, x², x, 1]`

In [ ]:
x = x_raw + delta * (np.random.rand(12) - 0.5)
y = x_raw**3 + delta * (np.random.rand(12) - 0.5)

M = np.vstack((x**3, x**2, x, np.ones(12))).T
s = np.linalg.lstsq(M, y, rcond=None)[0]

x_line = np.linspace(-15, 2, 100)
plt.figure(figsize=(9, 4))
plt.plot(x, y, 'D', label='Данные с шумом')
plt.plot(x_line, s[0]*x_line**3 + s[1]*x_line**2 + s[2]*x_line + s[3], 'r-', lw=2,
         label=f'y = {s[0]:.2f}x³ + {s[1]:.2f}x² + {s[2]:.2f}x + {s[3]:.2f}')
plt.title('Аппроксимация — 3-я степень')
plt.grid(); plt.legend(); plt.show()
print(f"Коэффициенты: a={s[0]:.4f}, b={s[1]:.4f}, c={s[2]:.4f}, d={s[3]:.4f}")

## 4. Нелинейная аппроксимация — `scipy.optimize.curve_fit`

Для нелинейных функций `lstsq` не подходит — используем итерационную оптимизацию.

$$f(x) = \beta_0 \cdot x^{\beta_1}$$

> **Важно:** начальное приближение `p0` критично для сходимости — без него алгоритм может найти неправильный знак.

In [ ]:
def power_func(x, b0, b1):
    return b0 * x ** b1

beta_true = (-5, 3)
xdata = np.linspace(1, 5, 100)
ydata = power_func(xdata, *beta_true) + 2.5 * np.random.randn(len(xdata))

# p0 — начальное приближение, помогает найти правильный знак
beta_opt, beta_cov = sp_optimize.curve_fit(power_func, xdata, ydata, p0=[-1, 1])
print(f"Истинные параметры:   b0={beta_true[0]}, b1={beta_true[1]}")
print(f"Найденные параметры:  b0={beta_opt[0]:.3f}, b1={beta_opt[1]:.3f}")

residuals = ydata - power_func(xdata, *beta_opt)
print(f"Квадратичное отклонение (RSS): {sum(residuals**2):.2f}")

fig, ax = plt.subplots(figsize=(9, 4))
ax.scatter(xdata, ydata, alpha=0.5, label='Данные с шумом')
ax.plot(xdata, power_func(xdata, *beta_true), 'r', lw=2, label='Истинная функция')
ax.plot(xdata, power_func(xdata, *beta_opt), 'b--', lw=2, label='Подобранная функция')
ax.set_xlabel('x'); ax.set_ylabel('f(x)')
ax.legend(); ax.grid(); plt.show()

## 5. Сравнение 1-й и 2-й степеней на одном графике

Используем `np.polyfit` — более компактный способ для полиномов.

In [ ]:
x = np.array([0.0, 0.2, 0.4, 0.6, 0.8, 1.0])
y = np.array([5.0, 5.0, 4.0, 4.0, 6.0, 6.0])

coef_1 = np.polyfit(x, y, 1)
coef_2 = np.polyfit(x, y, 2)
poly_1 = np.poly1d(coef_1)
poly_2 = np.poly1d(coef_2)

print(f"1-я степень: y = {coef_1[0]:.4f}x + {coef_1[1]:.4f}")
print(f"2-я степень: y = {coef_2[0]:.4f}x² + {coef_2[1]:.4f}x + {coef_2[2]:.4f}")

x_line = np.linspace(0, 1, 100)
plt.figure(figsize=(9, 4))
plt.scatter(x, y, color='black', s=80, zorder=5, label='Данные')
plt.plot(x_line, poly_1(x_line), 'b--', lw=2, label='1-я степень')
plt.plot(x_line, poly_2(x_line), 'r-', lw=2, label='2-я степень')
plt.title('Сравнение аппроксимаций (МНК)')
plt.xlabel('X'); plt.ylabel('Y')
plt.legend(); plt.grid(); plt.show()

## Итог

| Задача | Инструмент | Когда использовать |
|---|---|---|
| Полином любой степени | `np.linalg.lstsq` | Полный контроль над матрицей признаков |
| Полином (компактно) | `np.polyfit` | Быстро, без ручной матрицы |
| Нелинейная функция | `scipy.optimize.curve_fit` | Когда форма функции задана, но нелинейная |

**Ключевое правило:** чем выше степень полинома — тем лучше он подгоняется под обучающие данные, но хуже обобщается (переобучение).